# MESc files and the voltage pipeline

1. Open a `.mesc` file with `imread` and pick a unit.
2. Access metadata and raw data.
3. Run `mbo voltage` on it.
4. Open the `PF/traces/` outputs it writes.

In [ ]:
from pathlib import Path

import numpy as np
import mbo_utilities as mbo
from mbo_utilities.arrays import list_mesc_units

MESC_PATH = Path(r"X:/data/asako/stan112/stan112_expt12.mesc")

## Units in the file

A `.mesc` file can hold several measurement units (scans). `list_mesc_units` describes every one without opening the data; `imread` opens the first unit by default -- pass `unit=<index or "MSession_0/MUnit_3">` to choose another.

In [ ]:
units = list_mesc_units(MESC_PATH)
for u in units:
    print(u["index"], u["key"], u["modality_name"])

In [ ]:
arr = mbo.imread(MESC_PATH, unit=0)

print(f"Type: {type(arr).__name__}")
print(f"Shape (T, C, Z, Y, X): {arr.shape}")
print(f"Dtype: {arr.dtype}")
print(f"Unit: {arr.unit_key}, modality: {arr.modality}")

## Metadata

Canonical values (`fs`, `dx`, `dy`, `dz`, ...) are on the array directly. `arr.metadata` also carries MESc-specific extras under the `mesc_*` prefix -- z-axis meaning, ROI centroids, sync frame, RTMC curves.

In [ ]:
print(f"fs: {arr.fs} Hz")
print(f"dx, dy: {arr.dx}, {arr.dy} um")
print(f"num_timepoints: {arr.num_timepoints}")
print(f"num_zplanes: {arr.num_zplanes}")
print(f"Z axis meaning: {arr.metadata['mesc_z_axis_meaning']}")

In [ ]:
arr.metadata

## Raw data

Indexing takes 5D `(T, C, Z, Y, X)` keys. `np.asarray(arr)` gives one representative frame; use a slice for the data itself.

In [ ]:
frame = arr[0, 0, 0]  # one (Y, X) frame
print(f"frame shape: {frame.shape}")

chunk = arr[:100, 0, 0]  # first 100 timepoints, channel 0, plane/ROI 0
print(f"chunk shape: {chunk.shape}")

## Run the voltage pipeline

`mbo voltage` writes a `PF/` folder next to the `.mesc` file. `--init` writes a domain-grouping template to fill in first; a normal run reads it and produces the traces below.

```bash
mbo voltage stan112_expt12.mesc --init
mbo voltage stan112_expt12.mesc --domains PF/scanIDs_ROIs.pkl
```

## PF outputs

`PF/traces/` holds the plain outputs; everything else in `PF` is the archive format `mbo curate` opens.

| file | contents |
|---|---|
| `scans.csv` | one row per scan: id, MESc unit, frame rate, frames, ROIs |
| `domains.csv` | one row per domain: row index, name, ROI indices |
| `scan<id>_rois.npy` | raw mean fluorescence, shape (ROI, frame) |
| `scan<id>_dfof.npy` | dF/F per domain, shape (domain, frame) |
| `scan<id>_zscore.npy` | z-scored dF/F per domain, shape (domain, frame) |
| `scan<id>_denoised.npy` | wavelet-denoised trace per domain, shape (domain, frame) |
| `scan<id>_peaks.csv` | detected events: domain, frame, time in seconds |

Row `i` of every `(domain, frame)` array is row `i` of `domains.csv`.

In [ ]:
import pandas as pd

traces = MESC_PATH.parent / "PF" / "traces"

scans = pd.read_csv(traces / "scans.csv")
domains = pd.read_csv(traces / "domains.csv")
scans

In [ ]:
scan = str(scans.scan[0])
dfof = np.load(traces / f"scan{scan}_dfof.npy")
denoised = np.load(traces / f"scan{scan}_denoised.npy")
peaks = pd.read_csv(traces / f"scan{scan}_peaks.csv")

print(f"dfof: {dfof.shape}, denoised: {denoised.shape}")
peaks.groupby("domain").size().rename("n_events").to_frame()